<a href="https://colab.research.google.com/github/un1u3/devkota/blob/main/iteration6/tokenizer/notebooks/04_fst_on_full_cropse.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [18]:
# main fst algo
stopwords = {
    'अहिले', 'तेसैले', 'किनभने', 'त्यसैले', 'यसैले',
    'तेतिबेला', 'कहिले', 'पहिले', 'त्यो', 'यो', 'यी',
    'ती', 'जो', 'जति', 'सबै', 'कति', 'भोलि', 'हिजो',
    'आज', 'पहिलो', 'दोस्रो', 'तेस्रो', 'चाँडै', 'ढिलो',
    'तुरुन्त', 'फेरि', 'अझै', 'झन्', 'नि', 'पनि',
    'मलाई', 'मले',
    'उहाँले', 'उहाँलाई', 'उहाँको', 'उहाँ',
    'उसले', 'उसलाई', 'उसको',
    'तिमीले', 'तिमीलाई',
    'हामीले', 'हामीलाई',
    'उनले', 'उनलाई', 'उनको',
}

def fst(text, suffix_rules):
    words = text.split()
    result = []
    for word in words:
        if word in stopwords:
            result.append(word)
            continue
        marked = False
        for _, row in suffix_rules.iterrows():
            suffix  = row['suffix']
            if isinstance(suffix, str) and  word.endswith(suffix):
                # find where suffix starts
                root = word[:-len(suffix)]
                if len(root) > 0: # make sure root is not empty
                    marked_word = root + '<SLOT>' + suffix
                    result.append(marked_word)
                    marked = True
                    break
        if not marked:
            result.append(word) # no rule matched, keep as is
    return ' '.join(result)


def fst_fast(word, suffix_list):
    if word in stopwords:
        return word
    for suffix, slot in suffix_list:  # plain list, much faster
        if isinstance(suffix, str) and word.endswith(suffix):
            root = word[:-len(suffix)]
            if len(root) > 0:
                return root + '<SLOT>' + suffix
    return word



In [19]:
import pandas as pd
df =pd.read_csv('/content/drive/MyDrive/dataset-tokenizer/paradigm.csv')


def extract_suffix(stem, surface_form):
    if surface_form.startswith(stem):
        return surface_form[len(stem):]
    else:
        return None

df['suffix'] = df.apply(
    lambda row: extract_suffix(row['stem_nepali'], row['surface_form']),axis=1)

df
suffix_rules = df[['suffix','slot']].drop_duplicates()
suffix_rules = suffix_rules.sort_values('suffix',key = lambda x:x.str.len(), ascending=False)
suffix_rules = suffix_rules[suffix_rules['suffix'].str.len() > 0]
suffix_rules = suffix_rules.reset_index(drop=True)
suffix_list = [
    (row['suffix'], row['slot'])
    for _, row in suffix_rules.iterrows()
    if isinstance(row['suffix'], str)
]


In [5]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [2]:
# so the approach is
# instead of using it for whole dataset,to much operations
# we create a set of dataset,unique char only, then apply to it

In [8]:
unique_word = set()
with open('/content/drive/MyDrive/dataset-tokenizer/ne.txt','r', encoding ='utf-8') as f:
  for line in f:
    words= line.strip().split()
    for word in words:
      unique_word.add(word)


# total words 12,732,810
# unique 3571678


In [9]:
len(unique_word)

3571678

In [20]:
cache = {}
for i, word in enumerate(unique_word):
    cache[word] = fst_fast(word, suffix_list)
    if i % 100000 == 0:
        print(f"Progress: {i:,} / {len(unique_word):,}")

Progress: 0 / 3,571,678
Progress: 100,000 / 3,571,678
Progress: 200,000 / 3,571,678
Progress: 300,000 / 3,571,678
Progress: 400,000 / 3,571,678
Progress: 500,000 / 3,571,678
Progress: 600,000 / 3,571,678
Progress: 700,000 / 3,571,678
Progress: 800,000 / 3,571,678
Progress: 900,000 / 3,571,678
Progress: 1,000,000 / 3,571,678
Progress: 1,100,000 / 3,571,678
Progress: 1,200,000 / 3,571,678
Progress: 1,300,000 / 3,571,678
Progress: 1,400,000 / 3,571,678
Progress: 1,500,000 / 3,571,678
Progress: 1,600,000 / 3,571,678
Progress: 1,700,000 / 3,571,678
Progress: 1,800,000 / 3,571,678
Progress: 1,900,000 / 3,571,678
Progress: 2,000,000 / 3,571,678
Progress: 2,100,000 / 3,571,678
Progress: 2,200,000 / 3,571,678
Progress: 2,300,000 / 3,571,678
Progress: 2,400,000 / 3,571,678
Progress: 2,500,000 / 3,571,678
Progress: 2,600,000 / 3,571,678
Progress: 2,700,000 / 3,571,678
Progress: 2,800,000 / 3,571,678
Progress: 2,900,000 / 3,571,678
Progress: 3,000,000 / 3,571,678
Progress: 3,100,000 / 3,571,678
Pr

In [21]:
len(cache)

3571678

In [22]:
print(f"Cache built: {len(cache):,} entries")

# Check a few entries
test_words = ['गर्छु', 'खान्छु', 'गर्नुभयो', 'जान्छौं', 'मलाई']
for word in test_words:
    print(f"{word} → {cache.get(word, 'NOT FOUND')}")

Cache built: 3,571,678 entries
गर्छु → गर्<SLOT>छु
खान्छु → खा<SLOT>न्छु
गर्नुभयो → गर्<SLOT>नुभयो
जान्छौं → जा<SLOT>न्छौं
मलाई → मलाई


In [ ]:
corpus_path = '/content/drive/MyDrive/dataset-tokenizer/ne.txt'
output_path = '/content/drive/MyDrive/dataset-tokenizer/ne_marked.txt'

with open(corpus_path, 'r', encoding='utf-8') as fin, \
     open(output_path, 'w', encoding='utf-8') as fout:

    for i, line in enumerate(fin):
        words = line.strip().split()
        marked_words = [cache.get(word, word) for word in words]
        fout.write(' '.join(marked_words) + '\n')

        if i % 1000000 == 0:
            print(f"Progress: {i:,} / 12,732,810 lines")

print("Done! Marked corpus saved.")

Progress: 0 / 12,732,810 lines
Progress: 1,000,000 / 12,732,810 lines
Progress: 2,000,000 / 12,732,810 lines
Progress: 3,000,000 / 12,732,810 lines
Progress: 4,000,000 / 12,732,810 lines
Progress: 5,000,000 / 12,732,810 lines
Progress: 6,000,000 / 12,732,810 lines
Progress: 7,000,000 / 12,732,810 lines
Progress: 8,000,000 / 12,732,810 lines
Progress: 9,000,000 / 12,732,810 lines
Progress: 10,000,000 / 12,732,810 lines
Progress: 11,000,000 / 12,732,810 lines
